In [ ]:
# ML-Inter
# excercise-4 ( Pipelines , a simple way to keep you data preprocessing and modeling code organized )

# a pipeling bundles preprocessing and modeling steps so you can use the whole bundle as if it were a single step 

# data loding step

import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv('file_path')

y = data.Prices
X = data.drop(['Prices'] , axis = 1 )

X_train_full , X_val_full , y_train , y_val = train_test_split( X,y,train_size = 0.8 , test_size = 0.2 , random_state = 0 )

# selecing categorical columns ( with lower cardinality , for better results )

categorical_cols = [ col for col in data.columns if data[col].dtype == 'object' and data[col].nunique < 10 ]

# selecting numerical columns 

numerical_cols = [ col for col in data.column if data[col].dtype in ['int64' , 'float64']]

# keeping selected columns only 

my_cols = categorical_cols + numerical_cols
X_train = data[my_cols].copy()
X_val = data[my_cols].copy()


# construction of a full pipeline takes 3 steps 

# Step-1 , define preprocessing steps

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# simpilar to how pipeline bundles together data preprocessing and modeling steps , here ColumnTransformer class bundles different processing steps

numerical_transformer = SimpleImputer(strategy = 'constant')

categorical_transformer = Pipeline(steps=[
    ('impute' , SimpleImputer(strategy='most_frequent'))
    ('OneHot' , OneHotEncoder(handle_unknown = 'ignore'))
])

preprocessor = ColumnTransformer(
    transformers = [
        ('num' , numerical_transformer , numerical_cols)
        ('cat' , categorical_transformer , categorical_cols)
    ]
)


# Step-2 Define a model

from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators = 100 , random_state = 0)


# Step-3 Create and Evaluate Pipeling 
# with the pipeline , we preprocess the training data and fit the model in a single line of code
# we automatically preprocess validation and predict in a single line of code also 

from sklearn.metrics import mean_absolute_error

my_pipeline = Pipeline(steps = [
        ('preprocessor' , preprocessor)
        ('model' , model)    
])

my_pipeline.fit(X_train , y_train) # both preprocessing of X_train and fitting in single line of code

predictions = my_pipeline.predict(X_val) # both processing of X_val and model prediction in one step 

print('MAE for the pipeline is : ' , mean_absolute_error(y_val , predictions))
